In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
clinc_train = pd.read_csv("../clinc150_train_cleaned.csv")
clinc_val = pd.read_csv("../clinc150_validation_cleaned.csv")
clinc_test = pd.read_csv("../clinc150_test_cleaned.csv")

hwu_train = pd.read_csv("../hwu64_train_cleaned.csv")
hwu_test = pd.read_csv("../hwu64_test_cleaned.csv")

print("CLINC train:", len(clinc_train))
print("CLINC validation:", len(clinc_val))
print("CLINC test:", len(clinc_test))

print("HWU train:", len(hwu_train))
print("HWU test:", len(hwu_test))

CLINC train: 12600
CLINC validation: 2580
CLINC test: 4720
HWU train: 7600
HWU test: 912


In [3]:
hwu_train, hwu_val = train_test_split(
    hwu_train,
    test_size=0.15,
    random_state=42,
    stratify=hwu_train["intent"]
)

print("HWU train after split:", len(hwu_train))
print("HWU validation:", len(hwu_val))

HWU train after split: 6460
HWU validation: 1140


In [4]:
canonical_map = {
    "recipe": "recipe_request",
    "cooking_recipe": "recipe_request",

    "exchange_rate": "currency_information",
    "qa_currency": "currency_information",

    "definition": "definition_information",
    "qa_definition": "definition_information",

    "calculator": "math_calculation",
    "qa_maths": "math_calculation",

    "weather": "weather_information",
    "weather_query": "weather_information",

    "traffic": "traffic_information",
    "transport_traffic": "traffic_information",

    "calendar": "calendar_information",
    "calendar_query": "calendar_information",

    "alarm": "alarm_set",
    "alarm_set": "alarm_set",
}

for df in [clinc_train, clinc_val, clinc_test,
           hwu_train, hwu_val, hwu_test]:
    df["intent"] = df["intent"].fillna("out_of_scope").astype(str)
    df["intent"] = df["intent"].replace(canonical_map)

In [5]:
combined_train = pd.concat(
    [clinc_train, hwu_train],
    ignore_index=True
)

combined_validation = pd.concat(
    [clinc_val, hwu_val],
    ignore_index=True
)

combined_test = pd.concat(
    [clinc_test, hwu_test],
    ignore_index=True
)

print("Combined train:", len(combined_train))
print("Combined validation:", len(combined_validation))
print("Combined test:", len(combined_test))

print("\nTrain intents:", combined_train["intent"].nunique())
print("Validation intents:", combined_validation["intent"].nunique())
print("Test intents:", combined_test["intent"].nunique())

Combined train: 19060
Combined validation: 3720
Combined test: 5632

Train intents: 171
Validation intents: 171
Test intents: 171


In [63]:
REMOVE_INTENTS_MODEL = {"qa_factoid"}

combined_train = combined_train[
    ~combined_train["intent"].isin(REMOVE_INTENTS_MODEL)
].reset_index(drop=True)

combined_validation = combined_validation[
    ~combined_validation["intent"].isin(REMOVE_INTENTS_MODEL)
].reset_index(drop=True)

combined_test = combined_test[
    ~combined_test["intent"].isin(REMOVE_INTENTS_MODEL)
].reset_index(drop=True)

print("Train:", combined_train.shape)
print("Validation:", combined_validation.shape)
print("Test:", combined_test.shape)

print("\nqa_factoid present:")
print("Train:", "qa_factoid" in combined_train["intent"].values)
print("Validation:", "qa_factoid" in combined_validation["intent"].values)
print("Test:", "qa_factoid" in combined_test["intent"].values)

Train: (18927, 3)
Validation: (3697, 3)
Test: (5613, 3)

qa_factoid present:
Train: False
Validation: False
Test: False


In [6]:
print(combined_train.columns)

print("\nSample rows:")
print(combined_train[["utterance", "intent"]].head(10))

print("\nMissing values:")
print(combined_train[["utterance", "intent"]].isnull().sum())

Index(['utterance', 'label', 'intent'], dtype='str')

Sample rows:
                                          utterance               intent
0             can i make a reservation for redrobin  accept_reservations
1  is it possible to make a reservation at redrobin  accept_reservations
2                   does redrobin take reservations  accept_reservations
3                are reservations taken at redrobin  accept_reservations
4                     does redrobin do reservations  accept_reservations
5                        can acero take reservation  accept_reservations
6              can you make reservations at hodak's  accept_reservations
7       tell me if per se in nyc takes reservations  accept_reservations
8        tell me if the cheshire takes reservations  accept_reservations
9                      will qdoba take reservations  accept_reservations

Missing values:
utterance    0
intent       0
dtype: int64


In [7]:
X_train = combined_train["utterance"]
y_train = combined_train["intent"]

X_val = combined_validation["utterance"]
y_val = combined_validation["intent"]

X_test = combined_test["utterance"]
y_test = combined_test["intent"]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (19060,)
y_train: (19060,)
X_val: (3720,)
y_val: (3720,)
X_test: (5632,)
y_test: (5632,)


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)
X_test_tfidf = vectorizer.transform(X_test)

print("Training matrix:", X_train_tfidf.shape)
print("Validation matrix:", X_val_tfidf.shape)
print("Test matrix:", X_test_tfidf.shape)

Training matrix: (19060, 16328)
Validation matrix: (3720, 16328)
Test matrix: (5632, 16328)


In [9]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    C=5,
    random_state=42
)

model.fit(X_train_tfidf, y_train)

print("Model training complete!")

Model training complete!


In [10]:
y_val_pred = model.predict(X_val_tfidf)

print("Predictions generated:", len(y_val_pred))

Predictions generated: 3720


In [11]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy = accuracy_score(y_val, y_val_pred)

precision = precision_score(
    y_val,
    y_val_pred,
    average="macro",
    zero_division=0
)

recall = recall_score(
    y_val,
    y_val_pred,
    average="macro",
    zero_division=0
)

f1 = f1_score(
    y_val,
    y_val_pred,
    average="macro",
    zero_division=0
)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

Accuracy : 0.8403
Precision: 0.8605
Recall   : 0.8529
F1 Score : 0.8509


In [12]:
from sklearn.metrics import classification_report

report = classification_report(
    y_val,
    y_val_pred,
    zero_division=0,
    output_dict=True
)

report_df = pd.DataFrame(report).T

report_df = report_df.sort_values(
    by="f1-score"
)

report_df[["precision", "recall", "f1-score", "support"]].head(20)

,precision,recall,f1-score,support
qa_factoid,0.178571,0.217391,0.196078,23.0
out_of_scope,0.476190,0.200000,0.281690,100.0
datetime_query,0.400000,0.500000,0.444444,24.0
order,0.727273,0.400000,0.516129,20.0
travel_suggestion,0.588235,0.500000,0.540541,20.0
what_song,0.750000,0.450000,0.562500,20.0
audio_volume_up,0.500000,0.647059,0.564103,17.0
cancel,0.611111,0.550000,0.578947,20.0
calendar_set,0.619048,0.565217,0.590909,23.0
time,0.647059,0.550000,0.594595,20.0


In [13]:
from sklearn.metrics import confusion_matrix

labels = sorted(y_val.unique())

cm = confusion_matrix(
    y_val,
    y_val_pred,
    labels=labels
)

# Remove correct predictions from the matrix
cm_no_diag = cm.copy()
import numpy as np
np.fill_diagonal(cm_no_diag, 0)

# Find the largest confusion pairs
confusions = []

for i in range(len(labels)):
    for j in range(len(labels)):
        if cm_no_diag[i, j] > 0:
            confusions.append(
                (cm_no_diag[i, j], labels[i], labels[j])
            )

confusions = sorted(
    confusions,
    reverse=True
)

print("Top 20 confusion pairs:\n")

for count, actual, predicted in confusions[:20]:
    print(
        f"Actual: {actual:30} "
        f"Predicted: {predicted:30} "
        f"Count: {count}"
    )

Top 20 confusion pairs:

Actual: out_of_scope                   Predicted: qa_factoid                     Count: 13
Actual: out_of_scope                   Predicted: recipe_request                 Count: 10
Actual: what_song                      Predicted: music_query                    Count: 9
Actual: order                          Predicted: takeaway_order                 Count: 8
Actual: uber                           Predicted: transport_taxi                 Count: 7
Actual: time                           Predicted: datetime_query                 Count: 6
Actual: out_of_scope                   Predicted: news_query                     Count: 6
Actual: out_of_scope                   Predicted: definition_information         Count: 6
Actual: insurance                      Predicted: insurance_change               Count: 5
Actual: date                           Predicted: datetime_query                 Count: 5
Actual: change_volume                  Predicted: audio_volume_up        

In [14]:
error_df = pd.DataFrame({
    "utterance": X_val.values,
    "actual": y_val.values,
    "predicted": y_val_pred
})

errors = error_df[
    error_df["actual"] != error_df["predicted"]
]

print("Total validation errors:", len(errors))

print("\nExamples: what_song → music_query")
print(
    errors[
        (errors["actual"] == "what_song") &
        (errors["predicted"] == "music_query")
    ][["utterance", "actual", "predicted"]].to_string(index=False)
)

Total validation errors: 594

Examples: what_song → music_query
                                     utterance    actual   predicted
                         cool song, what is it what_song music_query
                            what music is this what_song music_query
              what am i listening to right now what_song music_query
what is the name of the song playing right now what_song music_query
                      what is that song called what_song music_query
           what's that song on the speaker now what_song music_query
 what's the name of the song playing right now what_song music_query
                what song is playing right now what_song music_query
          what's that song that is playing now what_song music_query


In [15]:
error_pairs = (
    errors.groupby(["actual", "predicted"])
    .size()
    .sort_values(ascending=False)
)

print("Top 20 error pairs:\n")
print(error_pairs.head(20))

Top 20 error pairs:

actual                predicted             
out_of_scope          qa_factoid                13
                      recipe_request            10
what_song             music_query                9
order                 takeaway_order             8
uber                  transport_taxi             7
out_of_scope          definition_information     6
time                  datetime_query             6
out_of_scope          news_query                 6
insurance             insurance_change           5
date                  datetime_query             5
change_volume         audio_volume_up            5
datetime_query        date                       4
alarm_set             alarm_query                4
shopping_list         shopping_list_update       4
last_maintenance      oil_change_when            4
social_post           social_query               4
book_hotel            car_rental                 4
music_likeness        music_query                4
improve_credit_s

In [19]:
import sentence_transformers

print("sentence-transformers:", sentence_transformers.__version__)

sentence-transformers: 6.0.1


In [20]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

sample_text = "what song is playing right now"

sample_embedding = embedding_model.encode(sample_text)

print("Embedding shape:", sample_embedding.shape)
print("First 10 values:", sample_embedding[:10])

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding shape: (384,)
First 10 values: [-0.01168271 -0.06972686  0.01203741 -0.01953224 -0.01007486  0.13530122
  0.04113213 -0.06586159 -0.00298466  0.03392861]


In [21]:
X_train_embeddings = embedding_model.encode(
    X_train.tolist(),
    show_progress_bar=True,
    batch_size=32
)

print("Training embeddings shape:", X_train_embeddings.shape)

Batches:   0%|          | 0/596 [00:00<?, ?it/s]

Training embeddings shape: (19060, 384)


In [22]:
X_val_embeddings = embedding_model.encode(
    X_val.tolist(),
    show_progress_bar=True,
    batch_size=32
)

X_test_embeddings = embedding_model.encode(
    X_test.tolist(),
    show_progress_bar=True,
    batch_size=32
)

print("Validation embeddings shape:", X_val_embeddings.shape)
print("Test embeddings shape:", X_test_embeddings.shape)

Batches:   0%|          | 0/117 [00:00<?, ?it/s]

Batches:   0%|          | 0/176 [00:00<?, ?it/s]

Validation embeddings shape: (3720, 384)
Test embeddings shape: (5632, 384)


In [23]:
embedding_classifier = LogisticRegression(
    max_iter=1000,
    C=5,
    random_state=42
)

embedding_classifier.fit(X_train_embeddings, y_train)

print("Embedding classifier training complete!")

Embedding classifier training complete!


In [24]:
y_val_pred_embeddings = embedding_classifier.predict(
    X_val_embeddings
)

print("Validation predictions generated:", len(y_val_pred_embeddings))

Validation predictions generated: 3720


In [25]:
accuracy_embeddings = accuracy_score(
    y_val,
    y_val_pred_embeddings
)

precision_embeddings = precision_score(
    y_val,
    y_val_pred_embeddings,
    average="macro",
    zero_division=0
)

recall_embeddings = recall_score(
    y_val,
    y_val_pred_embeddings,
    average="macro",
    zero_division=0
)

f1_embeddings = f1_score(
    y_val,
    y_val_pred_embeddings,
    average="macro",
    zero_division=0
)

print(f"Accuracy : {accuracy_embeddings:.4f}")
print(f"Precision: {precision_embeddings:.4f}")
print(f"Recall   : {recall_embeddings:.4f}")
print(f"F1 Score : {f1_embeddings:.4f}")

Accuracy : 0.8874
Precision: 0.9019
Recall   : 0.9021
F1 Score : 0.8988


In [26]:
error_df_embeddings = pd.DataFrame({
    "utterance": X_val.values,
    "actual": y_val.values,
    "predicted": y_val_pred_embeddings
})

errors_embeddings = error_df_embeddings[
    error_df_embeddings["actual"] != error_df_embeddings["predicted"]
]

print("Total validation errors:", len(errors_embeddings))

error_pairs_embeddings = (
    errors_embeddings
    .groupby(["actual", "predicted"])
    .size()
    .sort_values(ascending=False)
)

print("\nTop 20 error pairs:\n")
print(error_pairs_embeddings.head(20))

Total validation errors: 419

Top 20 error pairs:

actual                 predicted               
out_of_scope           qa_factoid                  15
                       definition_information      10
                       news_query                   9
what_song              music_query                  7
time                   datetime_query               7
datetime_query         date                         6
change_volume          audio_volume_up              6
uber                   transport_taxi               5
restaurant_suggestion  recommendation_locations     5
calendar_set           reminder_update              4
play_music             music_likeness               4
order                  takeaway_order               4
nutrition_info         recipe_request               4
music_query            what_song                    4
out_of_scope           recipe_request               4
                       qa_stock                     3
shopping_list          shopping_list_

In [27]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

C_values = [1, 2, 5, 10]

results = []

for C in C_values:
    clf = LogisticRegression(
        max_iter=1000,
        C=C,
        random_state=42
    )

    clf.fit(X_train_embeddings, y_train)

    val_pred = clf.predict(X_val_embeddings)

    accuracy = accuracy_score(y_val, val_pred)
    f1 = f1_score(
        y_val,
        val_pred,
        average="macro",
        zero_division=0
    )

    results.append({
        "C": C,
        "accuracy": accuracy,
        "macro_f1": f1
    })

results_df = pd.DataFrame(results)

print(results_df)

    C  accuracy  macro_f1
0   1  0.875538  0.882235
1   2  0.883871  0.893755
2   5  0.887366  0.898774
3  10  0.889516  0.901012


In [28]:
final_embedding_classifier = LogisticRegression(
    max_iter=1000,
    C=10,
    random_state=42
)

final_embedding_classifier.fit(
    X_train_embeddings,
    y_train
)

print("Final embedding classifier trained!")

Final embedding classifier trained!


In [29]:
y_test_pred = final_embedding_classifier.predict(
    X_test_embeddings
)

print("Test predictions generated:", len(y_test_pred))

Test predictions generated: 5632


In [30]:
test_accuracy = accuracy_score(
    y_test,
    y_test_pred
)

test_precision = precision_score(
    y_test,
    y_test_pred,
    average="macro",
    zero_division=0
)

test_recall = recall_score(
    y_test,
    y_test_pred,
    average="macro",
    zero_division=0
)

test_f1 = f1_score(
    y_test,
    y_test_pred,
    average="macro",
    zero_division=0
)

print(f"Test Accuracy : {test_accuracy:.4f}")
print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall   : {test_recall:.4f}")
print(f"Test F1 Score : {test_f1:.4f}")

Test Accuracy : 0.7797
Test Precision: 0.7986
Test Recall   : 0.8927
Test F1 Score : 0.8302


In [31]:
test_report = classification_report(
    y_test,
    y_test_pred,
    zero_division=0,
    output_dict=True
)

test_report_df = pd.DataFrame(test_report).T

test_report_df = test_report_df.sort_values(
    by="f1-score"
)

test_report_df[["precision", "recall", "f1-score", "support"]].head(20)

,precision,recall,f1-score,support
qa_factoid,0.105769,0.578947,0.178862,19.0
datetime_query,0.151515,0.263158,0.192308,19.0
out_of_scope,0.936652,0.207000,0.339066,1000.0
recommendation_locations,0.245614,0.736842,0.368421,19.0
news_query,0.285714,0.842105,0.426667,19.0
email_querycontact,0.318182,0.736842,0.444444,19.0
recommendation_movies,0.303030,1.000000,0.465116,10.0
play_audiobook,0.361702,0.894737,0.515152,19.0
qa_stock,0.372549,1.000000,0.542857,19.0
transport_query,0.416667,0.789474,0.545455,19.0


In [32]:
print("Actual test label distribution:")
print(y_test.value_counts().head(20))

print("\nActual out_of_scope count:")
print((y_test == "out_of_scope").sum())

Actual test label distribution:
intent
out_of_scope              1000
alarm_set                   49
calendar_information        49
definition_information      49
currency_information        49
play_music                  49
recipe_request              49
traffic_information         49
weather_information         49
math_calculation            44
accept_reservations         30
account_blocked             30
application_status          30
apr                         30
balance                     30
bill_balance                30
bill_due                    30
book_flight                 30
book_hotel                  30
calendar_update             30
Name: count, dtype: int64

Actual out_of_scope count:
1000


In [33]:
normal_mask = y_test != "out_of_scope"

normal_accuracy = accuracy_score(
    y_test[normal_mask],
    y_test_pred[normal_mask]
)

normal_f1 = f1_score(
    y_test[normal_mask],
    y_test_pred[normal_mask],
    average="macro",
    zero_division=0
)

print("Accuracy excluding OOS:", normal_accuracy)
print("Macro F1 excluding OOS:", normal_f1)

Accuracy excluding OOS: 0.9032815198618307
Macro F1 excluding OOS: 0.8875452526372224


In [34]:
print("OOS distribution:")
print("Train:", (y_train == "out_of_scope").sum())
print("Validation:", (y_val == "out_of_scope").sum())
print("Test:", (y_test == "out_of_scope").sum())

print("\nTotal samples:")
print("Train:", len(y_train))
print("Validation:", len(y_val))
print("Test:", len(y_test))

OOS distribution:
Train: 200
Validation: 100
Test: 1000

Total samples:
Train: 19060
Validation: 3720
Test: 5632


In [35]:
y_val_proba = final_embedding_classifier.predict_proba(X_val_embeddings)

max_val_proba = y_val_proba.max(axis=1)

confidence_df = pd.DataFrame({
    "actual": y_val.values,
    "predicted": y_val_pred,
    "confidence": max_val_proba
})

print("Average confidence:")
print(confidence_df.groupby("actual")["confidence"].mean().sort_values().head(20))

Average confidence:
actual
out_of_scope              0.463407
order                     0.524936
iot_hue_lighton           0.566779
nutrition_info            0.570262
qa_factoid                0.589088
travel_suggestion         0.614090
transport_query           0.626720
restaurant_suggestion     0.643104
todo_list                 0.643873
update_playlist           0.651704
shopping_list             0.654513
reminder_update           0.662127
shopping_list_update      0.666530
text                      0.673157
datetime_query            0.673800
change_volume             0.682011
confirm_reservation       0.682953
calendar_update           0.686728
time                      0.687645
restaurant_reservation    0.689681
Name: confidence, dtype: float32


In [36]:
import numpy as np
from sklearn.metrics import classification_report

thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]

threshold_results = []

for threshold in thresholds:
    adjusted_pred = y_val_pred_embeddings.copy()

    adjusted_pred[max_val_proba < threshold] = "out_of_scope"

    accuracy = accuracy_score(
        y_val,
        adjusted_pred
    )

    f1 = f1_score(
        y_val,
        adjusted_pred,
        average="macro",
        zero_division=0
    )

    oos_report = classification_report(
        y_val,
        adjusted_pred,
        output_dict=True,
        zero_division=0
    )

    threshold_results.append({
        "threshold": threshold,
        "accuracy": accuracy,
        "macro_f1": f1,
        "oos_precision": oos_report["out_of_scope"]["precision"],
        "oos_recall": oos_report["out_of_scope"]["recall"],
        "oos_f1": oos_report["out_of_scope"]["f1-score"]
    })

threshold_df = pd.DataFrame(threshold_results)

print(threshold_df.to_string(index=False))

 threshold  accuracy  macro_f1  oos_precision  oos_recall   oos_f1
      0.30  0.886828  0.902166       0.376623        0.58 0.456693
      0.35  0.882527  0.901143       0.302439        0.62 0.406557
      0.40  0.874462  0.898159       0.247148        0.65 0.358127
      0.45  0.864247  0.893064       0.205521        0.67 0.314554
      0.50  0.848925  0.884596       0.169082        0.70 0.272374
      0.55  0.834140  0.876052       0.142574        0.72 0.238017
      0.60  0.812634  0.861560       0.121951        0.75 0.209790


In [37]:
test_proba = final_embedding_classifier.predict_proba(X_test_embeddings)
test_max_proba = test_proba.max(axis=1)

y_test_adjusted = y_test_pred.copy()

y_test_adjusted[test_max_proba < 0.30] = "out_of_scope"

print("Adjusted test predictions generated:", len(y_test_adjusted))

Adjusted test predictions generated: 5632


In [38]:
final_test_accuracy = accuracy_score(
    y_test,
    y_test_adjusted
)

final_test_precision = precision_score(
    y_test,
    y_test_adjusted,
    average="macro",
    zero_division=0
)

final_test_recall = recall_score(
    y_test,
    y_test_adjusted,
    average="macro",
    zero_division=0
)

final_test_f1 = f1_score(
    y_test,
    y_test_adjusted,
    average="macro",
    zero_division=0
)

test_oos_report = classification_report(
    y_test,
    y_test_adjusted,
    output_dict=True,
    zero_division=0
)

print(f"Test Accuracy : {final_test_accuracy:.4f}")
print(f"Test Precision: {final_test_precision:.4f}")
print(f"Test Recall   : {final_test_recall:.4f}")
print(f"Test F1 Score : {final_test_f1:.4f}")

print("\nOOS metrics:")
print(f"Precision: {test_oos_report['out_of_scope']['precision']:.4f}")
print(f"Recall   : {test_oos_report['out_of_scope']['recall']:.4f}")
print(f"F1 Score : {test_oos_report['out_of_scope']['f1-score']:.4f}")

Test Accuracy : 0.8297
Test Precision: 0.8447
Test Recall   : 0.8830
Test F1 Score : 0.8543

OOS metrics:
Precision: 0.7976
Recall   : 0.5400
F1 Score : 0.6440


In [39]:
from sklearn.svm import LinearSVC

svm_classifier = LinearSVC(
    C=1.0,
    random_state=42
)

svm_classifier.fit(
    X_train_embeddings,
    y_train
)

print("Linear SVM training complete!")

Linear SVM training complete!


In [40]:
y_val_pred_svm = svm_classifier.predict(X_val_embeddings)

svm_accuracy = accuracy_score(
    y_val,
    y_val_pred_svm
)

svm_precision = precision_score(
    y_val,
    y_val_pred_svm,
    average="macro",
    zero_division=0
)

svm_recall = recall_score(
    y_val,
    y_val_pred_svm,
    average="macro",
    zero_division=0
)

svm_f1 = f1_score(
    y_val,
    y_val_pred_svm,
    average="macro",
    zero_division=0
)

print(f"SVM Accuracy : {svm_accuracy:.4f}")
print(f"SVM Precision: {svm_precision:.4f}")
print(f"SVM Recall   : {svm_recall:.4f}")
print(f"SVM F1 Score : {svm_f1:.4f}")

SVM Accuracy : 0.8879
SVM Precision: 0.8966
SVM Recall   : 0.9070
SVM F1 Score : 0.8979


In [41]:
from sklearn.neural_network import MLPClassifier

mlp_classifier = MLPClassifier(
    hidden_layer_sizes=(256,),
    activation="relu",
    solver="adam",
    alpha=1e-4,
    batch_size=64,
    learning_rate_init=1e-3,
    max_iter=40,
    early_stopping=True,
    validation_fraction=0.1,
    random_state=42
)

mlp_classifier.fit(
    X_train_embeddings,
    y_train
)

print("MLP training complete!")

MLP training complete!


In [42]:
y_val_pred_mlp = mlp_classifier.predict(X_val_embeddings)

mlp_accuracy = accuracy_score(
    y_val,
    y_val_pred_mlp
)

mlp_precision = precision_score(
    y_val,
    y_val_pred_mlp,
    average="macro",
    zero_division=0
)

mlp_recall = recall_score(
    y_val,
    y_val_pred_mlp,
    average="macro",
    zero_division=0
)

mlp_f1 = f1_score(
    y_val,
    y_val_pred_mlp,
    average="macro",
    zero_division=0
)

print(f"MLP Accuracy : {mlp_accuracy:.4f}")
print(f"MLP Precision: {mlp_precision:.4f}")
print(f"MLP Recall   : {mlp_recall:.4f}")
print(f"MLP F1 Score : {mlp_f1:.4f}")

MLP Accuracy : 0.8806
MLP Precision: 0.8955
MLP Recall   : 0.8949
MLP F1 Score : 0.8894


In [43]:
baseline_results = pd.DataFrame([
    {
        "model": "TF-IDF + Logistic Regression",
        "accuracy": 0.8403,
        "macro_f1": 0.8509
    },
    {
        "model": "MiniLM Embeddings + Logistic Regression",
        "accuracy": 0.8895,
        "macro_f1": 0.9010
    },
    {
        "model": "MiniLM Embeddings + Linear SVM",
        "accuracy": 0.8879,
        "macro_f1": 0.8979
    },
    {
        "model": "MiniLM Embeddings + MLP",
        "accuracy": 0.8806,
        "macro_f1": 0.8894
    }
])

baseline_results

,model,accuracy,macro_f1
0,TF-IDF + Logistic Regression,0.8403,0.8509
1,MiniLM Embeddings + Logistic Regression,0.8895,0.9010
2,MiniLM Embeddings + Linear SVM,0.8879,0.8979
3,MiniLM Embeddings + MLP,0.8806,0.8894


In [44]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.14.0
CUDA available: False


In [45]:
import time

from sentence_transformers import SentenceTransformer

candidate_models = [
    "all-MiniLM-L6-v2",
    "all-mpnet-base-v2",
    "multi-qa-MiniLM-L6-cos-v1"
]

benchmark_texts = X_train.sample(
    n=1000,
    random_state=42
).tolist()

for model_name in candidate_models:
    start = time.time()

    model = SentenceTransformer(model_name)

    embeddings = model.encode(
        benchmark_texts,
        batch_size=32,
        show_progress_bar=False
    )

    elapsed = time.time() - start

    print(
        f"{model_name:30} "
        f"shape={embeddings.shape} "
        f"time={elapsed:.2f}s"
    )

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

all-MiniLM-L6-v2               shape=(1000, 384) time=5.70s


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

multi-qa-MiniLM-L6-cos-v1      shape=(1000, 384) time=17.81s


In [46]:
import time
from sentence_transformers import SentenceTransformer

start = time.time()

model_mini = SentenceTransformer("all-MiniLM-L6-v2")

embeddings_mini = model_mini.encode(
    benchmark_texts,
    batch_size=32,
    show_progress_bar=False
)

elapsed = time.time() - start

print(
    "all-MiniLM-L6-v2",
    "shape =", embeddings_mini.shape,
    "time =", round(elapsed, 2), "s"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

all-MiniLM-L6-v2 shape = (1000, 384) time = 5.73 s


In [47]:
start = time.time()

model_mpnet = SentenceTransformer("all-mpnet-base-v2")

embeddings_mpnet = model_mpnet.encode(
    benchmark_texts,
    batch_size=32,
    show_progress_bar=False
)

elapsed = time.time() - start

print(
    "all-mpnet-base-v2",
    "shape =", embeddings_mpnet.shape,
    "time =", round(elapsed, 2), "s"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

all-mpnet-base-v2 shape = (1000, 768) time = 8.39 s


In [48]:
X_train_mpnet = model_mpnet.encode(
    X_train.tolist(),
    batch_size=32,
    show_progress_bar=True
)

X_val_mpnet = model_mpnet.encode(
    X_val.tolist(),
    batch_size=32,
    show_progress_bar=True
)

print("Train:", X_train_mpnet.shape)
print("Validation:", X_val_mpnet.shape)

Batches:   0%|          | 0/596 [00:00<?, ?it/s]

Batches:   0%|          | 0/117 [00:00<?, ?it/s]

Train: (19060, 768)
Validation: (3720, 768)


In [49]:
mpnet_classifier = LogisticRegression(
    max_iter=1000,
    C=10,
    random_state=42
)

mpnet_classifier.fit(
    X_train_mpnet,
    y_train
)

print("MPNet classifier training complete!")

MPNet classifier training complete!


In [50]:
y_val_pred_mpnet = mpnet_classifier.predict(X_val_mpnet)

mpnet_accuracy = accuracy_score(
    y_val,
    y_val_pred_mpnet
)

mpnet_f1 = f1_score(
    y_val,
    y_val_pred_mpnet,
    average="macro",
    zero_division=0
)

print(f"MPNet Accuracy: {mpnet_accuracy:.4f}")
print(f"MPNet Macro F1: {mpnet_f1:.4f}")

MPNet Accuracy: 0.8997
MPNet Macro F1: 0.9078


In [51]:
mpnet_C_values = [1, 2, 5, 10, 20]

mpnet_results = []

for C in mpnet_C_values:
    clf = LogisticRegression(
        max_iter=1000,
        C=C,
        random_state=42
    )

    clf.fit(X_train_mpnet, y_train)

    pred = clf.predict(X_val_mpnet)

    acc = accuracy_score(y_val, pred)
    f1 = f1_score(
        y_val,
        pred,
        average="macro",
        zero_division=0
    )

    mpnet_results.append({
        "C": C,
        "accuracy": acc,
        "macro_f1": f1
    })

mpnet_results_df = pd.DataFrame(mpnet_results)

print(mpnet_results_df.to_string(index=False))

 C  accuracy  macro_f1
 1  0.887634  0.892597
 2  0.895968  0.903288
 5  0.900000  0.907997
10  0.899731  0.907792
20  0.899194  0.906635


In [52]:
final_mpnet_classifier = LogisticRegression(
    max_iter=1000,
    C=5,
    random_state=42
)

final_mpnet_classifier.fit(
    X_train_mpnet,
    y_train
)

print("Final MPNet classifier trained!")

Final MPNet classifier trained!


In [53]:
mpnet_val_proba = final_mpnet_classifier.predict_proba(X_val_mpnet)
mpnet_val_max_proba = mpnet_val_proba.max(axis=1)

mpnet_val_pred = final_mpnet_classifier.predict(X_val_mpnet)

print("Validation predictions ready:", len(mpnet_val_pred))

Validation predictions ready: 3720


In [54]:
mpnet_thresholds = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]

mpnet_threshold_results = []

for threshold in mpnet_thresholds:
    adjusted_pred = mpnet_val_pred.copy()

    adjusted_pred[mpnet_val_max_proba < threshold] = "out_of_scope"

    accuracy = accuracy_score(
        y_val,
        adjusted_pred
    )

    macro_f1 = f1_score(
        y_val,
        adjusted_pred,
        average="macro",
        zero_division=0
    )

    oos_report = classification_report(
        y_val,
        adjusted_pred,
        output_dict=True,
        zero_division=0
    )

    mpnet_threshold_results.append({
        "threshold": threshold,
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "oos_precision": oos_report["out_of_scope"]["precision"],
        "oos_recall": oos_report["out_of_scope"]["recall"],
        "oos_f1": oos_report["out_of_scope"]["f1-score"]
    })

mpnet_threshold_df = pd.DataFrame(mpnet_threshold_results)

print(mpnet_threshold_df.to_string(index=False))

 threshold  accuracy  macro_f1  oos_precision  oos_recall   oos_f1
      0.20  0.900806  0.910930       0.473214        0.53 0.500000
      0.25  0.899731  0.910529       0.424658        0.62 0.504065
      0.30  0.896237  0.909021       0.356383        0.67 0.465278
      0.35  0.893280  0.908657       0.314655        0.73 0.439759
      0.40  0.885484  0.905449       0.262411        0.74 0.387435
      0.45  0.873925  0.899910       0.216524        0.76 0.337029
      0.50  0.853763  0.889175       0.173246        0.79 0.284173


In [55]:
X_test_mpnet = model_mpnet.encode(
    X_test.tolist(),
    batch_size=32,
    show_progress_bar=True
)

print("Test MPNet embeddings:", X_test_mpnet.shape)

Batches:   0%|          | 0/176 [00:00<?, ?it/s]

Test MPNet embeddings: (5632, 768)


In [56]:
mpnet_test_proba = final_mpnet_classifier.predict_proba(X_test_mpnet)

mpnet_test_max_proba = mpnet_test_proba.max(axis=1)

mpnet_test_pred = final_mpnet_classifier.predict(X_test_mpnet)

print("Test predictions ready:", len(mpnet_test_pred))

Test predictions ready: 5632


In [57]:
mpnet_test_pred_thresholded = mpnet_test_pred.copy()

mpnet_test_pred_thresholded[mpnet_test_max_proba < 0.20] = "out_of_scope"

In [58]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

test_accuracy = accuracy_score(y_test, mpnet_test_pred_thresholded)
test_precision = precision_score(
    y_test,
    mpnet_test_pred_thresholded,
    average="weighted",
    zero_division=0
)
test_recall = recall_score(
    y_test,
    mpnet_test_pred_thresholded,
    average="weighted",
    zero_division=0
)
test_f1 = f1_score(
    y_test,
    mpnet_test_pred_thresholded,
    average="macro",
    zero_division=0
)

oos_report = classification_report(
    y_test,
    mpnet_test_pred_thresholded,
    output_dict=True,
    zero_division=0
)

print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Weighted Precision: {test_precision:.4f}")
print(f"Weighted Recall: {test_recall:.4f}")
print(f"Macro F1: {test_f1:.4f}")

print("\nOOS Performance:")
print(f"OOS Precision: {oos_report['out_of_scope']['precision']:.4f}")
print(f"OOS Recall:    {oos_report['out_of_scope']['recall']:.4f}")
print(f"OOS F1:        {oos_report['out_of_scope']['f1-score']:.4f}")

Test Accuracy: 0.8320
Weighted Precision: 0.8645
Weighted Recall: 0.8320
Macro F1: 0.8609

OOS Performance:
OOS Precision: 0.8774
OOS Recall:    0.4580
OOS F1:        0.6018


In [59]:
test_report = classification_report(
    y_test,
    mpnet_test_pred_thresholded,
    output_dict=True,
    zero_division=0
)

test_report_df = pd.DataFrame(test_report).T

test_report_df = test_report_df[
    ~test_report_df.index.isin(["accuracy", "macro avg", "weighted avg"])
]

test_report_df = test_report_df.sort_values(
    "f1-score",
    ascending=True
)

print(test_report_df[["precision", "recall", "f1-score", "support"]].head(25).to_string())

                          precision    recall  f1-score  support
qa_factoid                 0.126437  0.578947  0.207547     19.0
datetime_query             0.266667  0.421053  0.326531     19.0
recommendation_locations   0.265306  0.684211  0.382353     19.0
news_query                 0.277778  0.789474  0.410959     19.0
smart_home                 0.571429  0.400000  0.470588     30.0
audio_volume_up            0.437500  0.538462  0.482759     13.0
change_volume              0.631579  0.400000  0.489796     30.0
calendar_set               0.600000  0.473684  0.529412     19.0
recommendation_movies      0.370370  1.000000  0.540541     10.0
music_query                0.500000  0.631579  0.558140     19.0
lists_createoradd          0.441176  0.789474  0.566038     19.0
play_audiobook             0.414634  0.894737  0.566667     19.0
audio_volume_down          0.461538  0.750000  0.571429      8.0
iot_hue_lighton            0.500000  0.666667  0.571429      3.0
out_of_scope             

In [60]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_test,
    mpnet_test_pred_thresholded,
    labels=sorted(y_test.unique())
)

labels = sorted(y_test.unique())

cm_df = pd.DataFrame(
    cm,
    index=labels,
    columns=labels
)

weak_intents = [
    "qa_factoid",
    "datetime_query",
    "recommendation_locations",
    "news_query",
    "smart_home",
    "change_volume",
    "audio_volume_up",
    "calendar_set",
    "music_query"
]

for intent in weak_intents:
    print(f"\n===== {intent} =====")

    row = cm_df.loc[intent].sort_values(ascending=False)

    print(
        row[row > 0].head(6).to_string()
    )


===== qa_factoid =====
qa_factoid            11
out_of_scope           5
email_querycontact     1
news_query             1
transport_query        1

===== datetime_query =====
datetime_query      8
date                5
time                3
timezone            1
next_holiday        1
datetime_convert    1

===== recommendation_locations =====
recommendation_locations    13
restaurant_suggestion        2
directions                   1
play_audiobook               1
recommendation_events        1
out_of_scope                 1

===== news_query =====
news_query      15
car_rental       1
out_of_scope     1
qa_factoid       1
calendar_set     1

===== smart_home =====
smart_home             12
recipe_request          7
iot_hue_lightdim        5
weather_information     2
iot_cleaning            1
play_radio              1

===== change_volume =====
change_volume        12
audio_volume_up       8
audio_volume_down     7
whisper_mode          2
play_audiobook        1

===== audio_volume_u

In [61]:
problem_intents = [
    "qa_factoid",
    "change_volume",
    "audio_volume_up",
    "audio_volume_down",
    "datetime_query",
    "date",
    "time"
]

for intent in problem_intents:
    print(f"\n{'='*60}")
    print(f"INTENT: {intent}")
    print(f"{'='*60}")

    examples = combined_test[
        combined_test["intent"] == intent
    ]["utterance"].head(10)

    for i, text in enumerate(examples, 1):
        print(f"{i}. {text}")


INTENT: qa_factoid
1. what year what the eiffel tower built
2. when does the super bowl officially start
3. how many months in a year
4. how many pages long is harry potter
5. where do cows come from
6. where is italy
7. give me the birth date of mahatma gandhi
8. which continent has highest growth of cotton and what is average production
9. when is shakira's birthday
10. please tell me about the historic facts about india

INTENT: change_volume
1. turn volume up to 4
2. adjust volume setting to 4
3. keep volume at 4 all the time
4. put volume setting on number 4
5. please make sure the volume stays at 4
6. put the volume to 4
7. volume to 4
8. turn down your speaker
9. decrease your decibel level
10. be more quiet

INTENT: audio_volume_up
1. could you speak a little more softly
2. turn it up
3. just increase the volume a little
4. olly turn the volume up
5. raise volume to level seven on music player
6. i want the speakers turned up high
7. rise volume
8. please increase the volume o

In [64]:
X_train = combined_train["utterance"]
y_train = combined_train["intent"]

X_val = combined_validation["utterance"]
y_val = combined_validation["intent"]

X_test = combined_test["utterance"]
y_test = combined_test["intent"]

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

print("\nUnique intents:")
print("Train:", y_train.nunique())
print("Validation:", y_val.nunique())
print("Test:", y_test.nunique())

Train: (18927,) (18927,)
Validation: (3697,) (3697,)
Test: (5613,) (5613,)

Unique intents:
Train: 170
Validation: 170
Test: 170


In [65]:
X_train_mpnet = model_mpnet.encode(
    X_train.tolist(),
    batch_size=32,
    show_progress_bar=True
)

X_val_mpnet = model_mpnet.encode(
    X_val.tolist(),
    batch_size=32,
    show_progress_bar=True
)

X_test_mpnet = model_mpnet.encode(
    X_test.tolist(),
    batch_size=32,
    show_progress_bar=True
)

print("Train embeddings:", X_train_mpnet.shape)
print("Validation embeddings:", X_val_mpnet.shape)
print("Test embeddings:", X_test_mpnet.shape)

Batches:   0%|          | 0/592 [00:00<?, ?it/s]

Batches:   0%|          | 0/116 [00:00<?, ?it/s]

Batches:   0%|          | 0/176 [00:00<?, ?it/s]

Train embeddings: (18927, 768)
Validation embeddings: (3697, 768)
Test embeddings: (5613, 768)


In [66]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score

final_mpnet_classifier = LogisticRegression(
    max_iter=1000,
    C=5,
    random_state=42
)

final_mpnet_classifier.fit(X_train_mpnet, y_train)

mpnet_val_pred = final_mpnet_classifier.predict(X_val_mpnet)

val_accuracy = accuracy_score(y_val, mpnet_val_pred)
val_macro_f1 = f1_score(
    y_val,
    mpnet_val_pred,
    average="macro",
    zero_division=0
)

print(f"Validation Accuracy: {val_accuracy:.4f}")
print(f"Validation Macro F1: {val_macro_f1:.4f}")

Validation Accuracy: 0.9061
Validation Macro F1: 0.9115


In [67]:
mpnet_val_proba = final_mpnet_classifier.predict_proba(X_val_mpnet)

mpnet_val_max_proba = mpnet_val_proba.max(axis=1)

mpnet_val_pred = final_mpnet_classifier.predict(X_val_mpnet)

print("Validation predictions ready:", len(mpnet_val_pred))

Validation predictions ready: 3697


In [68]:
mpnet_thresholds = [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]

mpnet_threshold_results = []

for threshold in mpnet_thresholds:
    adjusted_pred = mpnet_val_pred.copy()
    adjusted_pred[mpnet_val_max_proba < threshold] = "out_of_scope"

    accuracy = accuracy_score(y_val, adjusted_pred)
    macro_f1 = f1_score(
        y_val,
        adjusted_pred,
        average="macro",
        zero_division=0
    )

    oos_report = classification_report(
        y_val,
        adjusted_pred,
        output_dict=True,
        zero_division=0
    )

    mpnet_threshold_results.append({
        "threshold": threshold,
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "oos_precision": oos_report["out_of_scope"]["precision"],
        "oos_recall": oos_report["out_of_scope"]["recall"],
        "oos_f1": oos_report["out_of_scope"]["f1-score"]
    })

mpnet_threshold_df = pd.DataFrame(mpnet_threshold_results)

print(mpnet_threshold_df.to_string(index=False))

 threshold  accuracy  macro_f1  oos_precision  oos_recall   oos_f1
      0.15  0.907222  0.913722       0.643678        0.56 0.598930
      0.20  0.905870  0.913725       0.538462        0.63 0.580645
      0.25  0.903435  0.912311       0.468966        0.68 0.555102
      0.30  0.900189  0.911542       0.387097        0.72 0.503497
      0.35  0.896402  0.910835       0.333333        0.75 0.461538
      0.40  0.890181  0.908453       0.288321        0.79 0.422460
      0.45  0.879632  0.903534       0.238938        0.81 0.369021
      0.50  0.859345  0.892917       0.188764        0.84 0.308257


In [69]:
mpnet_test_proba = final_mpnet_classifier.predict_proba(X_test_mpnet)

mpnet_test_max_proba = mpnet_test_proba.max(axis=1)

mpnet_test_pred = final_mpnet_classifier.predict(X_test_mpnet)

mpnet_test_pred_thresholded = mpnet_test_pred.copy()
mpnet_test_pred_thresholded[mpnet_test_max_proba < 0.15] = "out_of_scope"

In [70]:
test_accuracy = accuracy_score(y_test, mpnet_test_pred_thresholded)

test_macro_f1 = f1_score(
    y_test,
    mpnet_test_pred_thresholded,
    average="macro",
    zero_division=0
)

test_report = classification_report(
    y_test,
    mpnet_test_pred_thresholded,
    output_dict=True,
    zero_division=0
)

print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Macro F1: {test_macro_f1:.4f}")

print("\nOOS Performance:")
print(f"OOS Precision: {test_report['out_of_scope']['precision']:.4f}")
print(f"OOS Recall:    {test_report['out_of_scope']['recall']:.4f}")
print(f"OOS F1:        {test_report['out_of_scope']['f1-score']:.4f}")

Test Accuracy: 0.8270
Test Macro F1: 0.8565

OOS Performance:
OOS Precision: 0.9320
OOS Recall:    0.4110
OOS F1:        0.5704


In [71]:
test_report_df = pd.DataFrame(test_report).T

test_report_df = test_report_df[
    ~test_report_df.index.isin(["accuracy", "macro avg", "weighted avg"])
]

test_report_df = test_report_df.sort_values(
    "f1-score",
    ascending=True
)

print(
    test_report_df[
        ["precision", "recall", "f1-score", "support"]
    ].head(20).to_string()
)

                          precision    recall  f1-score  support
datetime_query             0.258065  0.421053  0.320000     19.0
recommendation_locations   0.250000  0.736842  0.373333     19.0
news_query                 0.258621  0.789474  0.389610     19.0
smart_home                 0.545455  0.400000  0.461538     30.0
audio_volume_up            0.437500  0.538462  0.482759     13.0
change_volume              0.631579  0.400000  0.489796     30.0
lists_createoradd          0.357143  0.789474  0.491803     19.0
iot_hue_lighton            0.400000  0.666667  0.500000      3.0
recommendation_movies      0.344828  1.000000  0.512821     10.0
calendar_set               0.500000  0.526316  0.512821     19.0
play_audiobook             0.377778  0.894737  0.531250     19.0
music_query                0.480000  0.631579  0.545455     19.0
out_of_scope               0.931973  0.411000  0.570437   1000.0
audio_volume_down          0.461538  0.750000  0.571429      8.0
email_querycontact       

In [72]:
intent_list = sorted(y_train.unique())

print("Number of intents:", len(intent_list))

for i, intent in enumerate(intent_list, 1):
    print(f"{i:3}. {intent}")

Number of intents: 170
  1. accept_reservations
  2. account_blocked
  3. alarm_query
  4. alarm_remove
  5. alarm_set
  6. application_status
  7. apr
  8. audio_volume_down
  9. audio_volume_mute
 10. audio_volume_up
 11. balance
 12. bill_balance
 13. bill_due
 14. book_flight
 15. book_hotel
 16. calendar_information
 17. calendar_remove
 18. calendar_set
 19. calendar_update
 20. calories
 21. cancel
 22. cancel_reservation
 23. car_rental
 24. card_declined
 25. carry_on
 26. change_language
 27. change_volume
 28. confirm_reservation
 29. cook_time
 30. credit_limit
 31. credit_limit_change
 32. credit_score
 33. currency_information
 34. current_location
 35. damaged_card
 36. date
 37. datetime_convert
 38. datetime_query
 39. definition_information
 40. direct_deposit
 41. directions
 42. distance
 43. email_addcontact
 44. email_query
 45. email_querycontact
 46. email_sendemail
 47. expiration_date
 48. find_phone
 49. flight_status
 50. food_last
 51. freeze_account
 52. g

In [75]:
intent_family_map = {

    # Banking & Payments
    "balance": "banking_payments",
    "bill_balance": "banking_payments",
    "bill_due": "banking_payments",
    "direct_deposit": "banking_payments",
    "min_payment": "banking_payments",
    "pay_bill": "banking_payments",
    "payday": "banking_payments",
    "routing": "banking_payments",
    "transactions": "banking_payments",
    "transfer": "banking_payments",
    "spending_history": "banking_payments",

    # Cards & Security
    "account_blocked": "cards_security",
    "card_declined": "cards_security",
    "damaged_card": "cards_security",
    "freeze_account": "cards_security",
    "new_card": "cards_security",
    "pin_change": "cards_security",
    "report_fraud": "cards_security",
    "report_lost_card": "cards_security",
    "replacement_card_duration": "cards_security",
    "expiration_date": "cards_security",

    # Credit & Finance
    "apr": "credit_finance",
    "credit_limit": "credit_finance",
    "credit_limit_change": "credit_finance",
    "credit_score": "credit_finance",
    "improve_credit_score": "credit_finance",
    "interest_rate": "credit_finance",
    "income": "credit_finance",
    "taxes": "credit_finance",
    "w2": "credit_finance",

    # Rewards & Retirement
    "redeem_rewards": "rewards_retirement",
    "rewards_balance": "rewards_retirement",
    "rollover_401k": "rewards_retirement",

    # Insurance
    "insurance": "insurance",
    "insurance_change": "insurance",
    "international_fees": "insurance",

    # Travel
    "book_flight": "travel_flights",
    "flight_status": "travel_flights",
    "carry_on": "travel_flights",
    "lost_luggage": "travel_flights",
    "travel_alert": "travel_flights",
    "travel_notification": "travel_flights",
    "travel_suggestion": "travel_flights",
    "international_visa": "travel_flights",
    "plug_type": "travel_flights",

    # Hotels / Rental
    "book_hotel": "hotels_rental",
    "car_rental": "hotels_rental",

    # Restaurants / Ordering
    "accept_reservations": "restaurants_food",
    "cancel_reservation": "restaurants_food",
    "confirm_reservation": "restaurants_food",
    "restaurant_reservation": "restaurants_food",
    "restaurant_reviews": "restaurants_food",
    "restaurant_suggestion": "restaurants_food",
    "order": "restaurants_food",
    "order_status": "restaurants_food",
    "takeaway_order": "restaurants_food",
    "takeaway_query": "restaurants_food",
    "meal_suggestion": "restaurants_food",

    # Cooking / Nutrition
    "calories": "cooking_nutrition",
    "cook_time": "cooking_nutrition",
    "food_last": "cooking_nutrition",
    "ingredient_substitution": "cooking_nutrition",
    "ingredients_list": "cooking_nutrition",
    "nutrition_info": "cooking_nutrition",
    "recipe_request": "cooking_nutrition",

    # Calendar / Tasks
    "calendar_information": "calendar_tasks",
    "calendar_remove": "calendar_tasks",
    "calendar_set": "calendar_tasks",
    "calendar_update": "calendar_tasks",
    "meeting_schedule": "calendar_tasks",
    "reminder": "calendar_tasks",
    "reminder_update": "calendar_tasks",
    "schedule_meeting": "calendar_tasks",
    "timer": "calendar_tasks",
    "todo_list": "calendar_tasks",
    "todo_list_update": "calendar_tasks",

    # Date / Time
    "date": "date_time",
    "datetime_convert": "date_time",
    "datetime_query": "date_time",
    "next_holiday": "date_time",
    "time": "date_time",
    "timezone": "date_time",

    # Communication
    "email_addcontact": "communication",
    "email_query": "communication",
    "email_querycontact": "communication",
    "email_sendemail": "communication",
    "make_call": "communication",
    "text": "communication",

    # Lists
    "lists_createoradd": "lists_organization",
    "lists_query": "lists_organization",
    "lists_remove": "lists_organization",
    "shopping_list": "lists_organization",
    "shopping_list_update": "lists_organization",

    # Music / Audio
    "audio_volume_down": "music_audio",
    "audio_volume_mute": "music_audio",
    "audio_volume_up": "music_audio",
    "change_volume": "music_audio",
    "music_likeness": "music_audio",
    "music_query": "music_audio",
    "music_settings": "music_audio",
    "next_song": "music_audio",
    "play_audiobook": "music_audio",
    "play_music": "music_audio",
    "play_podcasts": "music_audio",
    "play_radio": "music_audio",
    "update_playlist": "music_audio",
    "what_song": "music_audio",
    "whisper_mode": "music_audio",

    # Smart Home
    "iot_cleaning": "smart_home_iot",
    "iot_coffee": "smart_home_iot",
    "iot_hue_lightchange": "smart_home_iot",
    "iot_hue_lightdim": "smart_home_iot",
    "iot_hue_lightoff": "smart_home_iot",
    "iot_hue_lighton": "smart_home_iot",
    "iot_hue_lightup": "smart_home_iot",
    "iot_wemo_off": "smart_home_iot",
    "iot_wemo_on": "smart_home_iot",
    "smart_home": "smart_home_iot",
    "sync_device": "smart_home_iot",
    "find_phone": "smart_home_iot",

    # Transport
    "current_location": "transport_navigation",
    "directions": "transport_navigation",
    "distance": "transport_navigation",
    "share_location": "transport_navigation",
    "traffic_information": "transport_navigation",
    "transport_query": "transport_navigation",
    "transport_taxi": "transport_navigation",
    "transport_ticket": "transport_navigation",
    "uber": "transport_navigation",

    # Entertainment / Social
    "play_game": "entertainment_social",
    "social_post": "entertainment_social",
    "social_query": "entertainment_social",

    # General / Other
    "change_language": "general_other",
    "definition_information": "general_other",
    "general_confirm": "general_other",
    "math_calculation": "general_other",
    "measurement_conversion": "general_other",
    "news_query": "general_other",
    "qa_stock": "general_other",
    "spelling": "general_other",
    "translate": "general_other",
    "weather_information": "general_other",
    "vaccines": "general_other",
    "gas": "general_other",
    "gas_type": "general_other",
    "how_busy": "general_other",
    "last_maintenance": "general_other",
    "jump_start": "general_other",
    "mpg": "general_other",
    "oil_change_how": "general_other",
    "oil_change_when": "general_other",
    "schedule_maintenance": "general_other",
    "tire_change": "general_other",
    "tire_pressure": "general_other",

    # OOS
    "out_of_scope": "out_of_scope",
    # Missing mappings

    "alarm_query": "calendar_tasks",
    "alarm_remove": "calendar_tasks",
    "alarm_set": "calendar_tasks",

    "application_status": "banking_payments",

    "cancel": "general_other",

    "currency_information": "general_other",

    "order_checks": "banking_payments",

    "pto_balance": "hr_workplace",
    "pto_request": "hr_workplace",
    "pto_request_status": "hr_workplace",
    "pto_used": "hr_workplace",

    "recommendation_events": "recommendations",
    "recommendation_locations": "recommendations",
    "recommendation_movies": "recommendations",

    "reset_settings": "device_settings",
}

In [76]:
all_intents = set(y_train.unique())
mapped_intents = set(intent_family_map.keys())

print("Total actual intents:", len(all_intents))
print("Total mapped intents:", len(mapped_intents))

print("\nMissing mappings:")
print(sorted(all_intents - mapped_intents))

print("\nExtra mappings:")
print(sorted(mapped_intents - all_intents))

print("\nNumber of families:", len(set(intent_family_map.values())))

Total actual intents: 170
Total mapped intents: 170

Missing mappings:
[]

Extra mappings:
[]

Number of families: 22


In [77]:
questionable_intents = [
    "application_status",
    "cancel",
    "currency_information",
    "order_checks",
    "pto_balance",
    "pto_request",
    "pto_request_status",
    "pto_used",
    "recommendation_events",
    "recommendation_locations",
    "recommendation_movies",
    "reset_settings"
]

for intent in questionable_intents:
    print(f"\n{'='*65}")
    print(f"INTENT: {intent}")
    print(f"FAMILY: {intent_family_map[intent]}")
    print(f"{'='*65}")

    examples = combined_train[
        combined_train["intent"] == intent
    ]["utterance"].head(8)

    for i, text in enumerate(examples, 1):
        print(f"{i}. {text}")


INTENT: application_status
FAMILY: banking_payments
1. when will my application for my credit card be processed
2. has the discover card approved my app
3. do you know if my amex card app went through
4. have they looked over my app for the new credit card yet
5. has my application for the mastercard card gone through
6. have my app for a new card been processed yet
7. has the application i put in for a new visa been processed
8. has there been any notice that my card app has been looked at

INTENT: cancel
FAMILY: general_other
1. please cancel what you are doing, i've changed my mind
2. never mind, cancel that
3. stop working on it, i need something else
4. cancel my last request, i know the answer
5. forget it, i do not need it anymore
6. pause
7. cancel that last thing
8. silence

INTENT: currency_information
FAMILY: general_other
1. how many pesos can i get for one dollar
2. what is the current going rate for exchanging dollars for pesos
3. tell me the exchange rate between dollar

In [78]:
intent_family_map["currency_information"] = "banking_payments"
intent_family_map["application_status"] = "cards_security"
intent_family_map["international_fees"] = "general_other"

print("Updated mappings:")
for intent in [
    "application_status",
    "currency_information",
    "international_fees",
    "order_checks",
    "cancel",
    "reset_settings"
]:
    print(f"{intent:25} -> {intent_family_map[intent]}")

Updated mappings:
application_status        -> cards_security
currency_information      -> banking_payments
international_fees        -> general_other
order_checks              -> banking_payments
cancel                    -> general_other
reset_settings            -> device_settings


In [79]:
mapped_families = pd.Series(
    y_train.map(intent_family_map)
)

print("Missing family labels:")
print(mapped_families.isna().sum())

print("\nFamily distribution:")
print(mapped_families.value_counts().sort_values(ascending=False))

Missing family labels:
0

Family distribution:
intent
general_other           2874
calendar_tasks          1752
music_audio             1704
banking_payments        1433
restaurants_food        1166
smart_home_iot          1150
transport_navigation    1122
cards_security          1100
credit_finance           900
travel_flights           900
cooking_nutrition        833
communication            659
lists_organization       601
date_time                593
hr_workplace             400
entertainment_social     397
recommendations          343
rewards_retirement       300
hotels_rental            200
insurance                200
out_of_scope             200
device_settings          100
Name: count, dtype: int64


In [80]:
# Create broad family labels from the intent labels

y_train_family = y_train.map(intent_family_map)
y_val_family = y_val.map(intent_family_map)
y_test_family = y_test.map(intent_family_map)

print("Train family labels:", y_train_family.shape)
print("Validation family labels:", y_val_family.shape)
print("Test family labels:", y_test_family.shape)

print("\nUnique families:")
print("Train:", y_train_family.nunique())
print("Validation:", y_val_family.nunique())
print("Test:", y_test_family.nunique())

print("\nMissing family labels:")
print("Train:", y_train_family.isna().sum())
print("Validation:", y_val_family.isna().sum())
print("Test:", y_test_family.isna().sum())

Train family labels: (18927,)
Validation family labels: (3697,)
Test family labels: (5613,)

Unique families:
Train: 22
Validation: 22
Test: 22

Missing family labels:
Train: 0
Validation: 0
Test: 0


In [81]:
family_classifier = LogisticRegression(
    max_iter=1000,
    C=5,
    random_state=42
)

family_classifier.fit(X_train_mpnet, y_train_family)

family_val_pred = family_classifier.predict(X_val_mpnet)

family_val_accuracy = accuracy_score(
    y_val_family,
    family_val_pred
)

family_val_macro_f1 = f1_score(
    y_val_family,
    family_val_pred,
    average="macro",
    zero_division=0
)

print(f"Family Validation Accuracy: {family_val_accuracy:.4f}")
print(f"Family Validation Macro F1: {family_val_macro_f1:.4f}")

Family Validation Accuracy: 0.9356
Family Validation Macro F1: 0.9195


In [82]:
from sklearn.metrics import confusion_matrix

family_labels = sorted(y_val_family.unique())

family_cm = confusion_matrix(
    y_val_family,
    family_val_pred,
    labels=family_labels
)

family_cm_df = pd.DataFrame(
    family_cm,
    index=family_labels,
    columns=family_labels
)

for family in family_labels:
    row = family_cm_df.loc[family].copy()
    row = row[row > 0].sort_values(ascending=False)

    print(f"\n===== {family} =====")
    print(row.head(5).to_string())


===== banking_payments =====
banking_payments    274
credit_finance        5
general_other         3
restaurants_food      1

===== calendar_tasks =====
calendar_tasks          309
lists_organization        4
recommendations           4
general_other             3
transport_navigation      3

===== cards_security =====
cards_security        219
rewards_retirement      1

===== communication =====
communication           118
general_other             2
entertainment_social      1
music_audio               1

===== cooking_nutrition =====
cooking_nutrition    162
general_other          2

===== credit_finance =====
credit_finance    178
cards_security      1
general_other       1

===== date_time =====
date_time         113
calendar_tasks      1
general_other       1

===== device_settings =====
device_settings    19
smart_home_iot      1

===== entertainment_social =====
entertainment_social    61
general_other            3
music_audio              3
recommendations          1
travel_f

In [83]:
train_in_domain = y_train_family != "out_of_scope"
val_in_domain = y_val_family != "out_of_scope"

family_classifier_indomain = LogisticRegression(
    max_iter=1000,
    C=5,
    random_state=42
)

family_classifier_indomain.fit(
    X_train_mpnet[train_in_domain],
    y_train_family[train_in_domain]
)

family_val_pred_indomain = family_classifier_indomain.predict(
    X_val_mpnet[val_in_domain]
)

family_val_accuracy_indomain = accuracy_score(
    y_val_family[val_in_domain],
    family_val_pred_indomain
)

family_val_macro_f1_indomain = f1_score(
    y_val_family[val_in_domain],
    family_val_pred_indomain,
    average="macro",
    zero_division=0
)

print(f"In-domain Family Accuracy: {family_val_accuracy_indomain:.4f}")
print(f"In-domain Family Macro F1: {family_val_macro_f1_indomain:.4f}")

In-domain Family Accuracy: 0.9561
In-domain Family Macro F1: 0.9527


In [84]:
from collections import defaultdict

family_intent_classifiers = {}
family_intents = {}

for family in sorted(y_train_family.unique()):

    # Skip OOS — it will be handled separately
    if family == "out_of_scope":
        continue

    family_mask = y_train_family == family

    X_family = X_train_mpnet[family_mask]
    y_family = y_train[family_mask]

    intents_in_family = sorted(y_family.unique())
    family_intents[family] = intents_in_family

    # A single-intent family does not need a classifier
    if len(intents_in_family) == 1:
        family_intent_classifiers[family] = intents_in_family[0]
        continue

    clf = LogisticRegression(
        max_iter=1000,
        C=5,
        random_state=42
    )

    clf.fit(X_family, y_family)

    family_intent_classifiers[family] = clf

print("Family-specific models created:", len(family_intent_classifiers))
print("\nIntents per family:")

for family in sorted(family_intents):
    print(
        f"{family:25} -> "
        f"{len(family_intents[family])} intents"
    )

Family-specific models created: 21

Intents per family:
banking_payments          -> 13 intents
calendar_tasks            -> 14 intents
cards_security            -> 11 intents
communication             -> 6 intents
cooking_nutrition         -> 7 intents
credit_finance            -> 9 intents
date_time                 -> 6 intents
device_settings           -> 1 intents
entertainment_social      -> 3 intents
general_other             -> 24 intents
hotels_rental             -> 2 intents
hr_workplace              -> 4 intents
insurance                 -> 2 intents
lists_organization        -> 5 intents
music_audio               -> 15 intents
recommendations           -> 3 intents
restaurants_food          -> 11 intents
rewards_retirement        -> 3 intents
smart_home_iot            -> 12 intents
transport_navigation      -> 9 intents
travel_flights            -> 9 intents


In [85]:
hierarchical_val_pred = []

for i in range(len(X_val_mpnet)):

    # Stage 1: predict family
    predicted_family = family_classifier.predict(
        X_val_mpnet[i].reshape(1, -1)
    )[0]

    # OOS family
    if predicted_family == "out_of_scope":
        final_intent = "out_of_scope"

    else:
        classifier_or_label = family_intent_classifiers[predicted_family]

        # Single-intent family
        if isinstance(classifier_or_label, str):
            final_intent = classifier_or_label

        # Multi-intent family
        else:
            final_intent = classifier_or_label.predict(
                X_val_mpnet[i].reshape(1, -1)
            )[0]

    hierarchical_val_pred.append(final_intent)

hierarchical_val_pred = np.array(hierarchical_val_pred)

hierarchical_val_accuracy = accuracy_score(
    y_val,
    hierarchical_val_pred
)

hierarchical_val_macro_f1 = f1_score(
    y_val,
    hierarchical_val_pred,
    average="macro",
    zero_division=0
)

print(f"Hierarchical Validation Accuracy: {hierarchical_val_accuracy:.4f}")
print(f"Hierarchical Validation Macro F1: {hierarchical_val_macro_f1:.4f}")

Hierarchical Validation Accuracy: 0.8888
Hierarchical Validation Macro F1: 0.8986


In [86]:
balanced_mpnet_classifier = LogisticRegression(
    max_iter=1000,
    C=5,
    class_weight="balanced",
    random_state=42
)

balanced_mpnet_classifier.fit(
    X_train_mpnet,
    y_train
)

balanced_val_pred = balanced_mpnet_classifier.predict(
    X_val_mpnet
)

balanced_val_accuracy = accuracy_score(
    y_val,
    balanced_val_pred
)

balanced_val_macro_f1 = f1_score(
    y_val,
    balanced_val_pred,
    average="macro",
    zero_division=0
)

print(f"Balanced Validation Accuracy: {balanced_val_accuracy:.4f}")
print(f"Balanced Validation Macro F1: {balanced_val_macro_f1:.4f}")

Balanced Validation Accuracy: 0.9042
Balanced Validation Macro F1: 0.9092


In [87]:
weak_intents = [
    "datetime_query",
    "recommendation_locations",
    "news_query",
    "smart_home",
    "audio_volume_up",
    "change_volume",
    "lists_createoradd",
    "calendar_set",
    "music_query"
]

weak_counts = (
    y_train.value_counts()
    .reindex(weak_intents)
    .sort_values()
)

print(weak_counts.to_string())

intent
audio_volume_up              96
smart_home                  100
change_volume               100
datetime_query              133
recommendation_locations    133
news_query                  134
lists_createoradd           134
calendar_set                134
music_query                 134


In [88]:
for intent in weak_intents:
    print(f"\n{'='*60}")
    print(f"INTENT: {intent}")
    print(f"{'='*60}")

    examples = combined_train[
        combined_train["intent"] == intent
    ]["utterance"].sample(
        n=min(10, (combined_train["intent"] == intent).sum()),
        random_state=42
    )

    for i, text in enumerate(examples, 1):
        print(f"{i}. {text}")


INTENT: datetime_query
1. what time of day is it in london
2. what time is it in mountain time zone
3. what is mondays actual date
4. show me the current time
5. time in chicago
6. twenty second april day
7. what date is the third friday of this month
8. i want exact time in washington right now
9. what's the time in dubai
10. what is the date easter falls on

INTENT: recommendation_locations
1. show shops around second street
2. show me thai food near me
3. where can i shop as a local tourist
4. can you find me a furniture store near me
5. local shops
6. give me the name of all the shops in my area
7. where can i get organic wheat
8. please recommend a restaurant in seattle
9. can you recommend a cheap restaurant in this area
10. wine shop

INTENT: news_query
1. can you search trump
2. what are the most recent headlines on cnn
3. did the yankees win last night
4. how is the news in ireland
5. dea guam prescription drug abuse on the rise
6. cnn headlines
7. keep me up to date on the e

In [90]:
import pandas as pd

audio_volume_up_aug = [
    "increase the volume",
    "make the sound louder",
    "turn the sound up",
    "raise the speaker volume",
    "make the audio louder",
    "turn up the speakers",
    "increase speaker volume",
    "make it louder please",
    "raise the audio level",
    "turn the volume higher",
    "please make the music louder",
    "can you turn the sound up",
    "increase the audio",
    "make the speakers louder",
    "raise the volume please",
    "turn up the audio",
    "speak louder",
    "make your voice louder",
    "increase the sound level",
    "turn the volume up",
    "please raise the volume",
    "could you make it louder",
    "turn the speakers up",
    "increase my volume",
    "raise my audio volume",
    "make the playback louder",
    "boost the speaker volume",
    "increase the playback volume",
    "make the music louder",
    "turn up my music",
    "raise the sound",
    "increase sound output",
    "make the audio louder please",
    "please turn it up",
    "can you raise the sound",
    "turn it up a little",
    "make the speakers louder please",
    "increase volume a little",
    "raise the volume a little",
    "turn the audio up"
]

change_volume_aug = [
    "set the volume to 2",
    "set the volume to 3",
    "set the volume to 4",
    "set the volume to 5",
    "set the volume to 6",
    "set the volume to 7",
    "set the speaker volume to 3",
    "set the speaker volume to 5",
    "set the speaker volume to 7",
    "change the volume to 2",
    "change the volume to 4",
    "change the volume to 6",
    "adjust the volume to 3",
    "adjust the volume to 5",
    "adjust the volume to 7",
    "put the volume at 2",
    "put the volume at 4",
    "put the volume at 6",
    "keep the volume at 3",
    "keep the volume at 5",
    "keep the volume at 7",
    "set audio level to 2",
    "set audio level to 4",
    "set audio level to 6",
    "change audio level to 3",
    "change audio level to 5",
    "change audio level to 7",
    "set playback volume to 2",
    "set playback volume to 4",
    "set playback volume to 6",
    "adjust speaker volume to 3",
    "adjust speaker volume to 5",
    "adjust speaker volume to 7",
    "change speaker volume to 2",
    "change speaker volume to 4",
    "change speaker volume to 6",
    "set the sound level to 3",
    "set the sound level to 5",
    "set the sound level to 7",
    "adjust the sound level to 4"
]

print("audio_volume_up:", len(audio_volume_up_aug))
print("change_volume:", len(change_volume_aug))

audio_volume_up: 40
change_volume: 40


In [91]:
volume_augmented = pd.DataFrame({
    "utterance": audio_volume_up_aug + change_volume_aug,
    "intent": (
        ["audio_volume_up"] * len(audio_volume_up_aug) +
        ["change_volume"] * len(change_volume_aug)
    ),
    "source": ["synthetic"] * (
        len(audio_volume_up_aug) + len(change_volume_aug)
    )
})

print("\nShape:", volume_augmented.shape)
print("\nClass counts:")
print(volume_augmented["intent"].value_counts())


Shape: (80, 3)

Class counts:
intent
audio_volume_up    40
change_volume      40
Name: count, dtype: int64


In [92]:
volume_aug_embeddings = model_mpnet.encode(
    volume_augmented["utterance"].tolist(),
    batch_size=32,
    show_progress_bar=True
)

print("Augmented embeddings:", volume_aug_embeddings.shape)

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Augmented embeddings: (80, 768)


In [93]:
import numpy as np

X_train_augmented = np.vstack([
    X_train_mpnet,
    volume_aug_embeddings
])

y_train_augmented = pd.concat([
    y_train.reset_index(drop=True),
    volume_augmented["intent"].reset_index(drop=True)
])

print("Original training:", X_train_mpnet.shape)
print("Augmented training:", X_train_augmented.shape)
print("Augmented labels:", y_train_augmented.shape)

print("\nNew class counts:")
print(y_train_augmented.value_counts().loc[
    ["audio_volume_up", "change_volume"]
])

Original training: (18927, 768)
Augmented training: (19007, 768)
Augmented labels: (19007,)

New class counts:
intent
audio_volume_up    136
change_volume      140
Name: count, dtype: int64


In [94]:
volume_aug_classifier = LogisticRegression(
    max_iter=1000,
    C=5,
    random_state=42
)

volume_aug_classifier.fit(
    X_train_augmented,
    y_train_augmented
)

volume_aug_val_pred = volume_aug_classifier.predict(
    X_val_mpnet
)

volume_aug_val_accuracy = accuracy_score(
    y_val,
    volume_aug_val_pred
)

volume_aug_val_macro_f1 = f1_score(
    y_val,
    volume_aug_val_pred,
    average="macro",
    zero_division=0
)

print(f"Augmented Validation Accuracy: {volume_aug_val_accuracy:.4f}")
print(f"Augmented Validation Macro F1: {volume_aug_val_macro_f1:.4f}")

Augmented Validation Accuracy: 0.9061
Augmented Validation Macro F1: 0.9112


In [95]:
from sklearn.preprocessing import normalize

X_train_mpnet_norm = normalize(X_train_mpnet)
X_val_mpnet_norm = normalize(X_val_mpnet)

print("Train:", X_train_mpnet_norm.shape)
print("Validation:", X_val_mpnet_norm.shape)

Train: (18927, 768)
Validation: (3697, 768)


In [96]:
import numpy as np

knn_val_pred = []

for i in range(len(X_val_mpnet_norm)):
    similarities = X_train_mpnet_norm @ X_val_mpnet_norm[i]
    nearest_idx = np.argmax(similarities)

    knn_val_pred.append(
        y_train.iloc[nearest_idx]
    )

knn_val_pred = np.array(knn_val_pred)

knn_accuracy = accuracy_score(y_val, knn_val_pred)
knn_macro_f1 = f1_score(
    y_val,
    knn_val_pred,
    average="macro",
    zero_division=0
)

print(f"1-NN Validation Accuracy: {knn_accuracy:.4f}")
print(f"1-NN Validation Macro F1: {knn_macro_f1:.4f}")

1-NN Validation Accuracy: 0.8353
1-NN Validation Macro F1: 0.8402


In [97]:
c_values_fine = [3, 4, 5, 6, 7, 8]

c_results = []

for c in c_values_fine:
    clf = LogisticRegression(
        max_iter=1000,
        C=c,
        random_state=42
    )

    clf.fit(X_train_mpnet, y_train)

    pred = clf.predict(X_val_mpnet)

    c_results.append({
        "C": c,
        "accuracy": accuracy_score(y_val, pred),
        "macro_f1": f1_score(
            y_val,
            pred,
            average="macro",
            zero_division=0
        )
    })

c_results_df = pd.DataFrame(c_results)

print(c_results_df.to_string(index=False))

 C  accuracy  macro_f1
 3  0.904517  0.909618
 4  0.905058  0.910297
 5  0.906140  0.911459
 6  0.906140  0.911413
 7  0.905870  0.911260
 8  0.906952  0.912586


In [98]:
candidate_mpnet_classifier = LogisticRegression(
    max_iter=1000,
    C=8,
    random_state=42
)

candidate_mpnet_classifier.fit(X_train_mpnet, y_train)

candidate_val_pred = candidate_mpnet_classifier.predict(X_val_mpnet)
candidate_val_proba = candidate_mpnet_classifier.predict_proba(X_val_mpnet)
candidate_val_max_proba = candidate_val_proba.max(axis=1)

print(
    f"Validation Accuracy: "
    f"{accuracy_score(y_val, candidate_val_pred):.4f}"
)

print(
    f"Validation Macro F1: "
    f"{f1_score(y_val, candidate_val_pred, average='macro', zero_division=0):.4f}"
)

Validation Accuracy: 0.9070
Validation Macro F1: 0.9126


In [99]:
candidate_mpnet_classifier = LogisticRegression(
    max_iter=1000,
    C=8,
    random_state=42
)

candidate_mpnet_classifier.fit(X_train_mpnet, y_train)

candidate_val_pred = candidate_mpnet_classifier.predict(X_val_mpnet)
candidate_val_proba = candidate_mpnet_classifier.predict_proba(X_val_mpnet)
candidate_val_max_proba = candidate_val_proba.max(axis=1)

print(
    f"Validation Accuracy: "
    f"{accuracy_score(y_val, candidate_val_pred):.4f}"
)

print(
    f"Validation Macro F1: "
    f"{f1_score(y_val, candidate_val_pred, average='macro', zero_division=0):.4f}"
)

Validation Accuracy: 0.9070
Validation Macro F1: 0.9126


In [100]:
candidate_thresholds = [
    0.10, 0.15, 0.20, 0.25,
    0.30, 0.35, 0.40, 0.45, 0.50
]

candidate_threshold_results = []

for threshold in candidate_thresholds:

    adjusted_pred = candidate_val_pred.copy()

    adjusted_pred[
        candidate_val_max_proba < threshold
    ] = "out_of_scope"

    accuracy = accuracy_score(
        y_val,
        adjusted_pred
    )

    macro_f1 = f1_score(
        y_val,
        adjusted_pred,
        average="macro",
        zero_division=0
    )

    oos_report = classification_report(
        y_val,
        adjusted_pred,
        output_dict=True,
        zero_division=0
    )

    candidate_threshold_results.append({
        "threshold": threshold,
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "oos_precision": oos_report["out_of_scope"]["precision"],
        "oos_recall": oos_report["out_of_scope"]["recall"],
        "oos_f1": oos_report["out_of_scope"]["f1-score"]
    })

candidate_threshold_df = pd.DataFrame(
    candidate_threshold_results
)

print(candidate_threshold_df.to_string(index=False))

 threshold  accuracy  macro_f1  oos_precision  oos_recall   oos_f1
      0.10  0.907222  0.912782       0.764706        0.52 0.619048
      0.15  0.907763  0.914139       0.670732        0.55 0.604396
      0.20  0.906681  0.914623       0.553398        0.57 0.561576
      0.25  0.906952  0.915084       0.511628        0.66 0.576419
      0.30  0.903706  0.913287       0.433121        0.68 0.529183
      0.35  0.900189  0.913014       0.364532        0.74 0.488449
      0.40  0.895862  0.912037       0.316667        0.76 0.447059
      0.45  0.890452  0.910377       0.279310        0.81 0.415385
      0.50  0.877468  0.902607       0.226519        0.82 0.354978


In [101]:
final_candidate_test_proba = candidate_mpnet_classifier.predict_proba(
    X_test_mpnet
)

final_candidate_test_max_proba = final_candidate_test_proba.max(axis=1)

final_candidate_test_pred = candidate_mpnet_classifier.predict(
    X_test_mpnet
)

final_candidate_test_pred_thresholded = final_candidate_test_pred.copy()

final_candidate_test_pred_thresholded[
    final_candidate_test_max_proba < 0.25
] = "out_of_scope"

In [102]:
final_candidate_test_report = classification_report(
    y_test,
    final_candidate_test_pred_thresholded,
    output_dict=True,
    zero_division=0
)

final_candidate_test_accuracy = accuracy_score(
    y_test,
    final_candidate_test_pred_thresholded
)

final_candidate_test_macro_f1 = f1_score(
    y_test,
    final_candidate_test_pred_thresholded,
    average="macro",
    zero_division=0
)

print(f"Test Accuracy: {final_candidate_test_accuracy:.4f}")
print(f"Test Macro F1: {final_candidate_test_macro_f1:.4f}")

print("\nOOS Performance:")
print(
    f"OOS Precision: "
    f"{final_candidate_test_report['out_of_scope']['precision']:.4f}"
)
print(
    f"OOS Recall: "
    f"{final_candidate_test_report['out_of_scope']['recall']:.4f}"
)
print(
    f"OOS F1: "
    f"{final_candidate_test_report['out_of_scope']['f1-score']:.4f}"
)

Test Accuracy: 0.8470
Test Macro F1: 0.8673

OOS Performance:
OOS Precision: 0.8841
OOS Recall: 0.5340
OOS F1: 0.6658


In [103]:
c_values_extended = [8, 10, 12, 15, 20]

extended_c_results = []

for c in c_values_extended:
    clf = LogisticRegression(
        max_iter=1000,
        C=c,
        random_state=42
    )

    clf.fit(X_train_mpnet, y_train)

    pred = clf.predict(X_val_mpnet)

    extended_c_results.append({
        "C": c,
        "accuracy": accuracy_score(y_val, pred),
        "macro_f1": f1_score(
            y_val,
            pred,
            average="macro",
            zero_division=0
        )
    })

extended_c_df = pd.DataFrame(extended_c_results)

print(extended_c_df.to_string(index=False))

 C  accuracy  macro_f1
 8  0.906952  0.912586
10  0.905329  0.910873
12  0.906411  0.912183
15  0.905058  0.910343
20  0.905058  0.910149


In [104]:
final_validation_grid = []

for c in [7, 8, 9]:
    clf = LogisticRegression(
        max_iter=1000,
        C=c,
        random_state=42
    )

    clf.fit(X_train_mpnet, y_train)

    val_pred = clf.predict(X_val_mpnet)
    val_proba = clf.predict_proba(X_val_mpnet)
    val_max_proba = val_proba.max(axis=1)

    for threshold in [0.20, 0.25, 0.30]:
        adjusted_pred = val_pred.copy()
        adjusted_pred[val_max_proba < threshold] = "out_of_scope"

        report = classification_report(
            y_val,
            adjusted_pred,
            output_dict=True,
            zero_division=0
        )

        final_validation_grid.append({
            "C": c,
            "threshold": threshold,
            "accuracy": accuracy_score(y_val, adjusted_pred),
            "macro_f1": f1_score(
                y_val,
                adjusted_pred,
                average="macro",
                zero_division=0
            ),
            "oos_precision": report["out_of_scope"]["precision"],
            "oos_recall": report["out_of_scope"]["recall"],
            "oos_f1": report["out_of_scope"]["f1-score"]
        })

final_validation_grid_df = pd.DataFrame(final_validation_grid)

print(
    final_validation_grid_df
    .sort_values("macro_f1", ascending=False)
    .to_string(index=False)
)

 C  threshold  accuracy  macro_f1  oos_precision  oos_recall   oos_f1
 8       0.25  0.906952  0.915084       0.511628        0.66 0.576419
 9       0.25  0.906681  0.914880       0.524194        0.65 0.580357
 8       0.20  0.906681  0.914623       0.553398        0.57 0.561576
 9       0.30  0.904247  0.914011       0.449664        0.67 0.538153
 9       0.20  0.906411  0.913850       0.593750        0.57 0.581633
 7       0.25  0.905329  0.913827       0.492537        0.66 0.564103
 7       0.20  0.906140  0.913772       0.561905        0.59 0.575610
 8       0.30  0.903706  0.913287       0.433121        0.68 0.529183
 7       0.30  0.902624  0.912184       0.429448        0.70 0.532319


In [105]:
print("Classifier:")
print(candidate_mpnet_classifier)

print("\nNumber of classes:")
print(len(candidate_mpnet_classifier.classes_))

print("\nFirst 10 classes:")
print(candidate_mpnet_classifier.classes_[:10])

print("\nOOS threshold:")
print(0.25)

Classifier:
LogisticRegression(C=8, max_iter=1000, random_state=42)

Number of classes:
170

First 10 classes:
['accept_reservations' 'account_blocked' 'alarm_query' 'alarm_remove'
 'alarm_set' 'application_status' 'apr' 'audio_volume_down'
 'audio_volume_mute' 'audio_volume_up']

OOS threshold:
0.25


In [106]:
import os
import joblib

MODEL_DIR = "../models"

os.makedirs(MODEL_DIR, exist_ok=True)

# Save the trained classifier
joblib.dump(
    candidate_mpnet_classifier,
    f"{MODEL_DIR}/intent_classifier.joblib"
)

# Save the configuration required during inference
model_config = {
    "embedding_model": "sentence-transformers/all-mpnet-base-v2",
    "oos_threshold": 0.25,
    "num_classes": 170,
    "model_type": "logistic_regression",
    "C": 8
}

joblib.dump(
    model_config,
    f"{MODEL_DIR}/model_config.joblib"
)

print("Saved files:")
print(f"- {MODEL_DIR}/intent_classifier.joblib")
print(f"- {MODEL_DIR}/model_config.joblib")

Saved files:
- ../models/intent_classifier.joblib
- ../models/model_config.joblib


In [107]:
print(os.listdir(MODEL_DIR))

['model_config.joblib', 'intent_classifier.joblib']
